# PV Project_Disproportionality Analysis

### Importing Libraries

In [1]:
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display
from pathlib import Path
import warnings
from scipy.special import gammaln, digamma, expit, logit
from scipy.optimize import minimize
from scipy.stats import gamma as G

### Establish DB connection

In [5]:
# Establishing connection wit the database

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DB_PATH = PROJECT_ROOT / "data" / "faers_db.duckdb"

con = duckdb.connect(str(DB_PATH), read_only=True) # read_only=True for exploration, and read_only=False only when creating tables.

### Identify Tables

In [6]:
pd.set_option("display.max_columns", 60, "display.width", 220, "display.max_colwidth", 50)
print(con.sql("SHOW TABLES").df())

                     name
0               case_base
1               case_drug
2           case_exposure
3              case_month
4            case_outcome
5                 case_pt
6             case_source
7              clean_demo
8                 demo_us
9             drug_detail
10               exposure
11             faers_flat
12                    m_N
13                   m_ev
14                  m_grp
15                 m_pair
16                 months
17            pair_counts
18              pair_grid
19             pair_month
20                ps_case
21          ps_case_event
22                ps_drug
23               raw_demo
24               raw_indi
25               raw_reac
26               raw_ther
27               ref_drug
28       ref_known_signal
29          signal_labels
30              signal_ts
31       study_case_event
32            study_cases
33        study_drug_rows
34   v_ingredient_summary
35   v_known_signal_check
36  v_known_signal_family
37        v_

### Explore the selected tables

In [7]:
pair_counts = con.sql("SELECT * FROM pair_counts").df()
pair_month= con.sql ("SELECT * FROM pair_month").df()
pair_grid= con.sql ("SELECT * FROM pair_grid").df()

In [ ]:
pair_counts.describe()
pair_counts

,a,n_exp,n_excl,N,n_event_all,n_event_excl
count,9251.000000,9251.000000,9251.000000,9251.0,9251.000000,9251.000000
mean,16.762296,12612.442979,3785.282132,5374222.0,7971.613447,6.709653
std,100.912119,7682.298922,2181.538927,0.0,22966.008795,24.825554
min,1.000000,358.000000,87.000000,5374222.0,1.000000,0.000000
25%,1.000000,4851.000000,2414.000000,5374222.0,275.000000,0.000000
50%,2.000000,11604.000000,3685.000000,5374222.0,1386.000000,1.000000
75%,6.000000,18865.000000,4520.000000,5374222.0,5492.000000,4.000000
max,3782.000000,23381.000000,7605.000000,5374222.0,411823.000000,1036.000000


In [10]:
print(len(pair_counts), "observed ingredient-event pairs")

9251 observed ingredient-event pairs


## Classical disproportionality analysis

* pair_counts table (9251 drug-event pairs, each against the full US background). 
*  Can be used as a proper signal table with four methods, a noise filter, seriousness, validation against the known signals, and sensitivity runs. 

#### 1. Pull the counts and compute PRR, ROR and IC

In [11]:

def disproportionality(pair_counts):
    df = pair_counts.copy()
    df["b"] = df.n_exp - df.a
    df["c"] = df.n_event_all - df.a - df.n_event_excl
    df["d"] = (df.N - df.n_exp - df.n_excl) - df.c
    a, b, c, d = [df[x].astype(float) for x in "abcd"]
    ok = (b > 0) & (c > 0) & (d > 0)                    # undefined otherwise; flagged, not dropped
    n = a + b + c + d
    df["E"] = (a + b) * (a + c) / n                      # expected count under independence

    with np.errstate(all="ignore"):
        df["PRR"]    = np.where(ok, (a/(a+b)) / (c/(c+d)), np.nan)
        se_prr       = np.sqrt(1/a - 1/(a+b) + 1/c - 1/(c+d))
        df["PRR_lo"] = np.exp(np.log(df.PRR) - 1.96*se_prr)
        df["chi2"]   = np.where(ok, (a*d - b*c)**2 * n / ((a+b)*(c+d)*(a+c)*(b+d)), np.nan)
        df["ROR"]    = np.where(ok, (a*d)/(b*c), np.nan)
        df["ROR_lo"] = np.exp(np.log(df.ROR) - 1.96*np.sqrt(1/a + 1/b + 1/c + 1/d))
        df["IC"]     = np.log2((a + 0.5) / (df.E + 0.5))
        df["IC025"]  = df.IC - 3.3*(a + 0.5)**-0.5 - 2*(a + 0.5)**-1.5   # Norén approximation
    return df

res = disproportionality(pair_counts)
print(len(res), "pairs;", res.PRR.isna().sum(), "with an empty cell (metrics undefined)")


9251 pairs; 25 with an empty cell (metrics undefined)


#### 2: EBGM prior, fitted on the full ingredient × event grid  (DuMouchel’s shrinkage)

EBGM shrinks small-count estimates toward 1. It fits a two-gamma prior to all observed pairs, then computes the posterior mean of the log relative reporting rate (EBGM) and its 5th percentile (EB05). Because only pairs with a ≥ 1 exist in your table, I fit a zero-truncated likelihood, so the prior isn’t biased by the missing zero cells.


In [ ]:
# Every ingredient x PT cell, including zero cells
grid = con.sql("""
WITH ev_tot AS (SELECT pt, COUNT(*) AS n_e FROM case_pt GROUP BY pt),
grp_n  AS (SELECT ingredient, g, COUNT(*) AS n FROM exposure GROUP BY 1, 2),
ev_grp AS (SELECT x.ingredient, x.g, p.pt, COUNT(*) AS n
           FROM exposure x JOIN case_pt p USING (caseid) GROUP BY 1, 2, 3),
ings AS (SELECT ingredient FROM grp_n WHERE g = 'exp'),
tot  AS (SELECT COUNT(DISTINCT caseid) AS N FROM case_pt)
SELECT i.ingredient, t.pt,
       COALESCE(a.n, 0)  AS a,
       ne.n              AS n_exp,
       COALESCE(nx.n, 0) AS n_excl,
       (SELECT N FROM tot) AS N,
       t.n_e             AS n_event_all,
       COALESCE(xe.n, 0) AS n_event_excl
FROM ings i CROSS JOIN ev_tot t
JOIN grp_n ne ON ne.ingredient = i.ingredient AND ne.g = 'exp'
LEFT JOIN grp_n nx ON nx.ingredient = i.ingredient AND nx.g = 'excl'
LEFT JOIN ev_grp a  ON a.ingredient  = i.ingredient AND a.pt  = t.pt AND a.g  = 'exp'
LEFT JOIN ev_grp xe ON xe.ingredient = i.ingredient AND xe.pt = t.pt AND xe.g = 'excl'
""").df()

grid["E"] = grid.n_exp * (grid.n_event_all - grid.n_event_excl) / (grid.N - grid.n_excl)
print("cells:", len(grid), "| observed:", int((grid.a > 0).sum()),
      "| E<=0 or NaN:", int((~(grid.E > 0)).sum()))

g = grid[grid.E > 0]
n_fit, E_fit = g.a.values.astype(float), g.E.values

def _nb_log(n, E, al, be):
    p = be / (be + E)
    return gammaln(al + n) - gammaln(al) - gammaln(n + 1) + al*np.log(p) + n*np.log1p(-p)

def nll_safe(th, n, E):
    a1, b1, a2, b2 = np.exp(th[:4]); w = expit(th[4])
    l = np.logaddexp(np.log(w) + _nb_log(n, E, a1, b1),
                     np.log1p(-w) + _nb_log(n, E, a2, b2))
    v = -l.sum()
    return v if np.isfinite(v) else 1e12

# order: log a1, log b1, log a2, log b2, logit w   (b1 may go very small)
bounds = [(-3, 3), (-8, 3), (-3, 3), (-3, 3), (-6, 6)]
starts = [np.r_[np.log([0.2, 0.1, 2.0, 4.0]), logit(1/3)],
          np.r_[np.log([0.5, 0.5, 1.5, 2.0]), 0.0],
          np.r_[np.log([0.1, 0.05, 3.0, 3.0]), logit(0.7)],
          np.r_[np.log([0.118, 0.0284, 0.793, 2.110]), logit(0.1355)]]   # your earlier solution

fits = [minimize(nll_safe, s, args=(n_fit, E_fit), method="L-BFGS-B", bounds=bounds) for s in starts]
for f in fits: print(round(f.fun, 1), f.success)
fit = min(fits, key=lambda r: r.fun)

a1, b1, a2, b2 = np.exp(fit.x[:4]); w = expit(fit.x[4])
prior = dict(alpha1=a1, beta1=b1, alpha2=a2, beta2=b2, w=w, nll=fit.fun, ok=fit.success)
print({k: round(float(v), 4) if not isinstance(v, (bool, np.bool_)) else bool(v) for k, v in prior.items()})
at_bound = [nm for nm, v, (lo, hi) in zip(["a1","b1","a2","b2","w"], fit.x, bounds)
            if abs(v - lo) < 1e-3 or abs(v - hi) < 1e-3]
print("parameters at a bound:", at_bound or "none")

cells: 119000 | observed: 9251 | E<=0 or NaN: 14
38967.3 True
38967.3 True
38967.3 True
38967.3 True
{'alpha1': 0.1184, 'beta1': 0.0284, 'alpha2': 0.7926, 'beta2': 2.11, 'w': 0.1355, 'nll': 38967.333, 'ok': True}
parameters at a bound: none


In [13]:
# Apply the prior (EBGM and EB05)

def ebgm_apply(df, a1, b1, a2, b2, w):
    n, E = df.a.values.astype(float), df.E.values
    valid = E > 0
    nn, EE = n[valid], E[valid]
    l1 = np.log(w)    + _nb_log(nn, EE, a1, b1)
    l2 = np.log1p(-w) + _nb_log(nn, EE, a2, b2)
    Q  = np.exp(l1 - np.logaddexp(l1, l2))
    ebgm_v = np.exp(Q*(digamma(a1 + nn) - np.log(b1 + EE)) + (1 - Q)*(digamma(a2 + nn) - np.log(b2 + EE)))
    lo, hi = np.zeros_like(nn), np.full_like(nn, 1e3)
    for _ in range(60):                       # bisection for the 5th percentile
        mid = (lo + hi) / 2
        cdf = Q*G.cdf(mid, a1 + nn, scale=1/(b1 + EE)) + (1 - Q)*G.cdf(mid, a2 + nn, scale=1/(b2 + EE))
        lo, hi = np.where(cdf < 0.05, mid, lo), np.where(cdf < 0.05, hi, mid)
    ebgm = np.full(len(df), np.nan); eb05 = np.full(len(df), np.nan)
    ebgm[valid], eb05[valid] = ebgm_v, (lo + hi) / 2
    return ebgm, eb05

res["EBGM"], res["EB05"] = ebgm_apply(res, a1, b1, a2, b2, w)
res["raw_rr"] = res.a / res.E

# Check: shrinkage should be strong at small a and fade as a grows
chk = res.dropna(subset=["EBGM", "raw_rr"]).copy()
chk["ratio"] = chk.EBGM / chk.raw_rr
chk["a_bin"] = pd.cut(chk.a, [0, 2, 4, 9, 29, 1e9])
print(chk.groupby("a_bin", observed=True).ratio.median().round(2))



a_bin
(0.0, 2.0]              0.41
(2.0, 4.0]              0.75
(4.0, 9.0]              0.86
(9.0, 29.0]             0.94
(29.0, 1000000000.0]    0.99
Name: ratio, dtype: float64


#### 3: Signal criteria and noise filter

In [14]:

NOISE = (r"DRUG INEFFECTIVE|PRODUCT USE|OFF LABEL|INCORRECT (?:DOSE|DRUG|ROUTE)|WRONG|"
         r"DEVICE|PRODUCT (?:QUALITY|CONTAMINATION|PACKAGING|STORAGE|COUNTERFEIT)|"
         r"INTENTIONAL|ACCIDENTAL|MEDICATION ERROR|NO ADVERSE EVENT|INJECTION SITE|"
         r"EXPIRED|DRUG DOSE OMISSION|UNEVALUABLE|TREATMENT NONCOMPLIANCE|UNDERDOSE|OVERDOSE|"
         r"DRUG INTERACTION|LACK OF EFFICACY|BLOOD GLUCOSE (?:INCREASED|ABNORMAL|DECREASED|FLUCTUATION)")
res["is_noise"] = res.pt.str.contains(NOISE, regex=True, na=False)
print(int(res.is_noise.sum()), "noise pairs of", len(res))

def flag(df, min_a=3):
    df = df.copy()
    df["evans"]  = (df.a >= min_a) & (df.PRR >= 2) & (df.chi2 >= 4)          # PRR criterion
    df["ror_sig"] = (df.a >= min_a) & (df.ROR_lo > 1)
    df["ic_sig"]  = (df.a >= min_a) & (df.IC025 > 0)
    df["eb_sig"]  = (df.a >= min_a) & (df.EB05 >= 2)                         # DuMouchel's usual cut
    df["n_methods"] = df[["evans","ror_sig","ic_sig","eb_sig"]].sum(axis=1)
    df["signal"]  = df.evans & df.ror_sig & df.ic_sig                        # same rule as signal_ts
    df["strong"]  = df.signal & df.eb_sig
    return df

sig = flag(res, 3)
cols = ["evans", "ror_sig", "ic_sig", "eb_sig", "signal", "strong"]
print(sig.groupby("ingredient")[cols].sum())
print("\nafter removing noise PTs:")
print(sig[~sig.is_noise].groupby("ingredient")[["signal", "strong"]].sum())

# Small-count pairs: shrinkage should cut the number passing
small = sig[(sig.a <= 4) & ~sig.is_noise]
print(f"\n{len(small)} pairs with a<=4 | raw RR>=2: {(small.a/small.E >= 2).sum()} | EB05>=2: {(small.EB05 >= 2).sum()}")

436 noise pairs of 9251
               evans  ror_sig  ic_sig  eb_sig  signal  strong
ingredient                                                   
canagliflozin    284      302     268     197     251     197
dapagliflozin    199      213     183     123     172     123
dulaglutide      156      176     145      96     132      96
empagliflozin    183      193     167     120     162     120
ertugliflozin     21       23      22      14      21      14
liraglutide      160      186     152      96     136      96
semaglutide       98      107      85      59      81      59

after removing noise PTs:
               signal  strong
ingredient                   
canagliflozin     246     193
dapagliflozin     164     117
dulaglutide        93      65
empagliflozin     158     117
ertugliflozin      17      12
liraglutide       117      85
semaglutide        78      57

6357 pairs with a<=4 | raw RR>=2: 2457 | EB05>=2: 114


The “blood glucose” and “drug interaction” terms considered as  noise for this analysis.


In [ ]:
# noise 
print(sig[sig.is_noise & sig.signal].sort_values("a", ascending=False)[["ingredient","pt","a"]].head(40))

         ingredient                             pt     a
2764    dulaglutide        BLOOD GLUCOSE INCREASED  2930
2408    dulaglutide            INJECTION SITE PAIN  2928
3666    dulaglutide             DRUG DOSE OMISSION  1298
492     dulaglutide     INJECTION SITE HAEMORRHAGE  1135
483     dulaglutide    INCORRECT DOSE ADMINISTERED  1008
2754    liraglutide        BLOOD GLUCOSE INCREASED   858
4866    dulaglutide           ACCIDENTAL UNDERDOSE   709
4637    dulaglutide                      UNDERDOSE   694
488     dulaglutide        INJECTION SITE BRUISING   649
4180  canagliflozin        BLOOD GLUCOSE INCREASED   602
467     dulaglutide        BLOOD GLUCOSE DECREASED   567
4210    dulaglutide        INJECTION SITE ERYTHEMA   531
3211    dulaglutide            ACCIDENTAL OVERDOSE   501
5341    dulaglutide                       OVERDOSE   484
2753    dulaglutide            INJECTION SITE MASS   459
4011    dulaglutide        INJECTION SITE PRURITUS   439
2140  dapagliflozin        BLOO

#### 4: Seriousness and the ranked signal list

Seriousness comes from the existing view and is case-level & not event level

In [16]:
ser = con.sql("SELECT * FROM v_pair_serious").df()
lab = con.sql("SELECT ingredient, pt, onset, status FROM signal_labels").df()

out = (sig.merge(ser[["ingredient", "pt", "serious_cases", "deaths", "hospitalised", "life_threat"]],
                 on=["ingredient", "pt"], how="left")
          .merge(lab, on=["ingredient", "pt"], how="left"))
out["pct_serious"] = (100 * out.serious_cases / out.a).round(1)

# 1. Noise-flagged signals: check what the filter removes
print("=== Noise-flagged signals (top 40) ===")
display(out[out.is_noise & out.signal].sort_values("a", ascending=False)
        [["ingredient", "pt", "a"]].head(40))

# 2. Weak tier: signals rejected by EB05
weak = out[out.signal & ~out.eb_sig & ~out.is_noise]
print(f"\n=== Weak tier: {len(weak)} signals, median a = {weak.a.median()} ===")
display(weak.sort_values("a", ascending=False)[["ingredient", "pt", "a", "PRR", "IC025", "EB05"]].head(20).round(2))

# 3. Strong signals per drug
top = out[out.strong & ~out.is_noise].sort_values(["ingredient", "EB05"], ascending=[True, False])
show = ["pt", "a", "PRR", "ROR_lo", "IC025", "EBGM", "EB05", "pct_serious", "deaths", "status"]
for ing, d in top.groupby("ingredient"):
    print(f"\n=== {ing}: {len(d)} strong signals (non-noise) ===")
    display(d[show].head(15).round(2))

=== Noise-flagged signals (top 40) ===


,ingredient,pt,a
2764,dulaglutide,BLOOD GLUCOSE INCREASED,2930
2408,dulaglutide,INJECTION SITE PAIN,2928
3666,dulaglutide,DRUG DOSE OMISSION,1298
492,dulaglutide,INJECTION SITE HAEMORRHAGE,1135
483,dulaglutide,INCORRECT DOSE ADMINISTERED,1008
2754,liraglutide,BLOOD GLUCOSE INCREASED,858
4866,dulaglutide,ACCIDENTAL UNDERDOSE,709
4637,dulaglutide,UNDERDOSE,694
488,dulaglutide,INJECTION SITE BRUISING,649
4180,canagliflozin,BLOOD GLUCOSE INCREASED,602



=== Weak tier: 227 signals, median a = 8.0 ===


,ingredient,pt,a,PRR,IC025,EB05
513,dulaglutide,INAPPROPRIATE SCHEDULE OF DRUG ADMINISTRATION,459,2.13,0.93,1.95
1802,liraglutide,WEIGHT INCREASED,265,2.07,0.84,1.84
4249,liraglutide,HEART RATE INCREASED,110,2.23,0.83,1.84
4202,canagliflozin,RENAL INJURY,87,2.13,0.72,1.71
3667,canagliflozin,NEPHROLITHIASIS,84,2.39,0.88,1.91
3789,canagliflozin,CYSTITIS,66,2.33,0.79,1.79
1625,dapagliflozin,RENAL FAILURE,63,2.26,0.74,1.73
5169,canagliflozin,HAEMATURIA,58,2.19,0.67,1.65
3370,canagliflozin,URINARY INCONTINENCE,52,2.36,0.75,1.74
2840,dulaglutide,INCREASED APPETITE,47,2.32,0.70,1.68



=== canagliflozin: 193 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
2179,LIMB AMPUTATION,233,6608.77,3552.65,7.21,263.70,236.46,85.0,3,emerging
1816,DIABETIC FOOT INFECTION,688,1383.99,1197.61,7.53,233.81,219.51,91.4,6,emerging
3806,OSTEOMYELITIS ACUTE,231,2047.51,1431.84,7.13,242.15,217.03,84.0,1,emerging
288,METATARSAL EXCISION,99,9360.06,2983.21,6.52,254.99,215.46,83.8,0,emerging
4546,TOE AMPUTATION,1915,763.95,779.01,7.55,206.89,199.23,75.5,15,emerging
3264,FOOT AMPUTATION,517,819.22,710.18,7.31,208.85,194.17,85.3,7,emerging
2512,GAS GANGRENE,95,1496.98,908.87,6.39,222.42,187.27,97.9,0,emerging
3268,DIABETIC FOOT,683,466.81,428.32,7.17,175.66,164.88,85.7,6,emerging
2493,AMPUTATION,285,495.93,415.14,6.91,177.63,160.97,60.4,4,emerging
2415,OSTEOMYELITIS CHRONIC,95,626.64,439.17,6.26,184.36,155.22,83.2,2,emerging


... (8 outputs left out)


=== liraglutide: 85 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
412,WEIGHT LOSS POOR,219,157.91,137.89,6.32,115.93,103.59,0.9,0,emerging
2911,MEDULLARY THYROID CANCER,19,132.85,79.86,4.06,87.75,59.09,36.8,1,pre-existing
2880,PANCREATIC CARCINOMA STAGE IV,16,121.04,69.87,3.79,80.15,51.97,87.5,9,pre-existing
3239,DRUG DOSE TITRATION NOT PERFORMED,68,70.84,55.16,5.01,59.66,48.65,10.3,0,emerging
4533,PANCREATIC CARCINOMA,349,59.47,54.74,5.44,52.52,48.05,69.1,203,pre-existing
4910,PAPILLARY THYROID CANCER,44,73.04,53.32,4.72,60.19,46.63,36.4,0,pre-existing
128,PANCREATITIS,1109,53.83,55.67,5.46,48.24,45.90,41.9,19,pre-existing
1462,PANCREATIC CARCINOMA METASTATIC,68,43.34,33.98,4.55,38.84,31.67,79.4,44,pre-existing
4572,PANCREATITIS ACUTE,169,36.72,31.81,4.70,33.82,29.74,72.8,6,pre-existing
3728,ERUCTATION,294,32.46,29.52,4.66,30.26,27.47,3.1,0,emerging



=== semaglutide: 57 strong signals (non-noise) ===


,pt,a,PRR,ROR_lo,IC025,EBGM,EB05,pct_serious,deaths,status
7314,RETINOPATHY,19,85.79,54.66,3.97,72.30,48.69,5.3,0,emerging
5682,INTERCEPTED PRODUCT DISPENSING ERROR,11,106.79,58.60,3.22,78.19,46.07,0.0,0,emerging
9033,PRODUCT COMMUNICATION ISSUE,10,104.08,55.45,3.05,74.87,42.89,20.0,0,emerging
8617,DRUG DOSE TITRATION NOT PERFORMED,14,69.05,40.79,3.45,57.53,36.14,0.0,0,emerging
8433,DRUG TITRATION ERROR,16,52.47,32.15,3.50,45.98,29.82,0.0,0,emerging
1196,ERUCTATION,68,37.73,30.51,4.48,36.40,29.68,4.4,0,emerging
7044,INTERCEPTED DRUG DISPENSING ERROR,14,43.09,25.52,3.22,37.96,23.84,0.0,0,emerging
5268,DIABETIC RETINOPATHY,11,46.42,25.65,2.93,39.38,23.20,0.0,0,emerging
7307,INTERCEPTED PRODUCT PRESCRIBING ERROR,5,68.99,28.40,1.70,44.97,19.87,0.0,0,emerging
5142,PRODUCT DISPENSING ERROR,41,23.89,17.83,3.70,23.07,17.70,2.4,0,emerging


#### 5: Validate against the known signals

In [17]:
known = [("canagliflozin", "AMPUTATION"), ("canagliflozin", "FOURNIER"), ("canagliflozin", "KETOACIDOSIS"),
         ("dapagliflozin", "FOURNIER"),   ("dapagliflozin", "KETOACIDOSIS"),
         ("empagliflozin", "FOURNIER"),   ("empagliflozin", "KETOACIDOSIS"),
         ("liraglutide", "PANCREATITIS"), ("dulaglutide", "PANCREATITIS"), ("semaglutide", "PANCREATITIS")]

rows = []
for ing, rx in known:
    pool = sig[(sig.ingredient == ing) & (sig.a >= 3) & ~sig.is_noise]      # ranking pool for this drug
    hits = pool[pool.pt.str.contains(rx, regex=True)]
    for _, r in hits.iterrows():
        rows.append(dict(ingredient=ing, family=rx, pt=r.pt, a=int(r.a), IC025=round(r.IC025, 2),
                         EB05=round(r.EB05, 2), methods=int(r.n_methods),
                         rank_by_EB05=int((pool.EB05 > r.EB05).sum() + 1), of=len(pool)))
val = pd.DataFrame(rows).sort_values(["ingredient", "family", "rank_by_EB05"])
display(val)

# Sensitivity: share of known families with at least one fully flagged PT
fam = val.groupby(["ingredient", "family"]).methods.max().reset_index()
print("known families flagged by all four methods:", int((fam.methods == 4).sum()), "of", len(known))

# Lead times against FDA dates
display(con.sql("SELECT * FROM v_known_signal_family ORDER BY ingredient, pt_regex").df())

,ingredient,family,pt,a,IC025,EB05,methods,rank_by_EB05,of
0,canagliflozin,AMPUTATION,LIMB AMPUTATION,233,7.21,236.46,4,1,878
4,canagliflozin,AMPUTATION,TOE AMPUTATION,1915,7.55,199.23,4,5,878
2,canagliflozin,AMPUTATION,FOOT AMPUTATION,517,7.31,194.17,4,6,878
1,canagliflozin,AMPUTATION,AMPUTATION,285,6.91,160.97,4,9,878
3,canagliflozin,AMPUTATION,LEG AMPUTATION,668,6.89,133.30,4,12,878
6,canagliflozin,AMPUTATION,FINGER AMPUTATION,16,3.14,17.68,4,67,878
5,canagliflozin,AMPUTATION,HAND AMPUTATION,3,0.52,7.21,4,112,878
7,canagliflozin,FOURNIER,FOURNIER'S GANGRENE,126,6.08,103.00,4,20,878
9,canagliflozin,KETOACIDOSIS,DIABETIC KETOACIDOSIS,2589,6.95,127.99,4,14,878
8,canagliflozin,KETOACIDOSIS,KETOACIDOSIS,461,6.71,122.58,4,16,878


known families flagged by all four methods: 10 of 10


,ingredient,pt_regex,reg_date,first_pt,status,first_detect,lead_days
0,canagliflozin,AMPUTATION,2016-05-18,TOE AMPUTATION,emerging,2016-06-30,-43
1,canagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-11-30,-93
2,canagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,pre-existing,2014-02-28,441
3,dapagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-12-31,-124
4,dapagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,pre-existing,2014-09-30,227
5,dulaglutide,PANCREATITIS,NaT,PANCREATITIS,emerging,2015-05-31,<NA>
6,empagliflozin,FOURNIER,2018-08-29,FOURNIER'S GANGRENE,emerging,2018-10-31,-63
7,empagliflozin,KETOACIDOSIS,2015-05-15,DIABETIC KETOACIDOSIS,emerging,2015-02-28,76
8,liraglutide,PANCREATITIS,NaT,PANCREATITIS,pre-existing,2014-01-31,<NA>
9,semaglutide,PANCREATITIS,NaT,PANCREATITIS,emerging,2018-09-30,<NA>


#### 6: Sensitivity analyses

A signal is more convincing when it survives each of these changes. Run them with the same functions:

In [18]:
variants = {"main (a>=3)": flag(res, 3),
            "a>=5": flag(res, 5),
            "no noise PTs": flag(res[~res.is_noise], 3)}

def keyset(df): return set(zip(df[df.signal].ingredient, df[df.signal].pt))
base = keyset(variants["main (a>=3)"])
summ = pd.DataFrame({k: {"signals": len(keyset(v)),
                         "overlap_with_main": len(keyset(v) & base),
                         "jaccard": round(len(keyset(v) & base) / max(len(keyset(v) | base), 1), 2)}
                     for k, v in variants.items()}).T
display(summ)

# Pairs that were flagged by the monthly series at some point, but not signalling at end of 2019
ever = lab[lab.status != "never"].merge(sig[["ingredient", "pt", "signal"]], on=["ingredient", "pt"], how="left")
lost = ever[~ever.signal.fillna(False).astype(bool)]
print(len(ever), "ever signalled |", len(lost), "not signalling at end of 2019")
print(lost.groupby("ingredient").size())

,signals,overlap_with_main,jaccard
main (a>=3),955.0,955.0,1.00
a>=5,819.0,819.0,0.86
no noise PTs,873.0,873.0,0.91


1138 ever signalled | 183 not signalling at end of 2019
ingredient
canagliflozin    73
dapagliflozin    15
dulaglutide      31
empagliflozin    28
ertugliflozin     2
liraglutide      26
semaglutide       8
dtype: int64


### Close the DB connection

In [16]:
con.close()

# Explicitly close any ambient DuckDB connections
try:
    duckdb.close()
    print("✓ Closed ambient DuckDB connection.")
except Exception as e:
    print(f"Notice: {e}")

✓ Closed ambient DuckDB connection.
